# Nobeyama Radioheliograph — Implementation / 구현

**Paper #29** — Nakajima et al., *The Nobeyama Radioheliograph*, Proc. IEEE **82**(5), 705-713 (1994). DOI: [10.1109/5.284737](https://doi.org/10.1109/5.284737)

## Overview / 개요

**English.** This notebook reproduces the key radio-interferometer physics of NoRH: (1) the T-shaped array geometry and its u-v coverage, (2) the synthesized beam (dirty beam / PSF) from the u-v sampling function, (3) a forward-modeled NoRH dirty/clean image of a synthetic flare, (4) gyrosynchrotron spectra at 17 and 34 GHz for different magnetic fields and electron power-law indices, and (5) a simple 1D profile of the solar disk at 17 GHz.

**한국어.** 이 노트북은 NoRH의 핵심 전파 간섭계 물리를 재현한다: (1) T자형 배열 기하와 u-v 덮개, (2) u-v 샘플링 함수로부터의 합성 빔 (dirty beam / PSF), (3) 합성 플레어의 순방향 모델 NoRH dirty/clean 영상, (4) 다양한 자기장과 전자 멱법칙 지수에 대한 17 및 34 GHz 자이로싱크로트론 스펙트럼, (5) 17 GHz 태양 원반의 단순 1D 프로파일.

## 1. Imports and Setup / 임포트 및 설정

**English.** Standard scientific Python stack. All code comments and docstrings follow Google style, English-only.

**한국어.** 표준 과학 파이썬 스택. 모든 코드 주석과 독스트링은 Google 스타일 영문을 따른다.

In [ ]:
"""Setup: imports and constants for NoRH reproduction."""

import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fft2, fftshift, ifft2

# Physical constants (SI)
c = 2.99792458e8  # m/s
k_B = 1.380649e-23  # J/K
e = 1.602176634e-19  # C
m_e = 9.1093837015e-31  # kg

# NoRH parameters
NU_17 = 17.0e9  # Hz
NU_34 = 34.0e9  # Hz
LAMBDA_17 = c / NU_17  # ~ 0.01763 m
D_FUND = 1.528  # m, fundamental spacing
N_ANT_TOTAL = 84
ARM_EW = 488.96  # m
ARM_NS = 220.06  # m

np.random.seed(42)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10

print(f"Wavelength at 17 GHz: {LAMBDA_17*1e2:.4f} cm")
print(f"Theoretical resolution (E-W): {LAMBDA_17/ARM_EW * 206265:.2f} arcsec")
print(f"Theoretical FOV from d={D_FUND} m: {LAMBDA_17/D_FUND * 206265/60:.2f} arcmin")

## 2. T-shaped Array Geometry / T자형 배열 기하

**English.** We construct a simplified but faithful T-array: eighty-four antennas distributed between the E-W arm (488.96 m) and N-S arm (220.06 m), with increasing spacings $d, 2d, 4d, 8d, 16d$ moving outward from the center.

**한국어.** 단순화되지만 충실한 T자형 배열을 구성한다: 동서 암 (488.96 m)과 남북 암 (220.06 m) 사이에 분배된 84개 안테나와, 중심으로부터 바깥으로 이동하며 증가하는 간격 $d, 2d, 4d, 8d, 16d$.

In [ ]:
def build_t_array(d=D_FUND, arm_ew=ARM_EW, arm_ns=ARM_NS, n_total=N_ANT_TOTAL):
    """Construct NoRH-like T-array antenna positions.

    The spacing pattern is a geometric progression of d, 2d, 4d, 8d, 16d that repeats
    until the arm length is filled. Half of the antennas populate each arm, with one
    shared antenna at the array phase center.

    Args:
        d: Fundamental antenna spacing in meters.
        arm_ew: East-West arm total length in meters (half each side).
        arm_ns: North-South arm length in meters (extends only north from center).
        n_total: Target total number of antennas.

    Returns:
        Nx2 numpy array of (x, y) antenna positions in meters with origin at phase center.
    """
    spacings = [1, 2, 4, 8, 16]

    def pack_arm(length_max):
        positions = [0.0]
        x = 0.0
        i = 0
        while True:
            step = spacings[i % len(spacings)] * d
            x_next = x + step
            if x_next > length_max:
                break
            positions.append(x_next)
            x = x_next
            i += 1
        return np.array(positions)

    ew_pos = pack_arm(arm_ew / 2.0)
    ew_positions = np.concatenate([-ew_pos[1:][::-1], ew_pos])
    ns_pos = pack_arm(arm_ns)

    ant = []
    for x in ew_positions:
        ant.append([x, 0.0])
    for y in ns_pos[1:]:
        ant.append([0.0, y])
    ant = np.array(ant)

    if len(ant) > n_total:
        ant = ant[:n_total]
    return ant


antennas = build_t_array()
print(f"Built T-array with {len(antennas)} antennas.")
print(f"E-W extent: {antennas[:, 0].min():.1f} -> {antennas[:, 0].max():.1f} m")
print(f"N-S extent: {antennas[:, 1].min():.1f} -> {antennas[:, 1].max():.1f} m")

In [ ]:
# Visualize the array / 배열 시각화
fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.scatter(antennas[:, 0], antennas[:, 1], s=30, c="#1f77b4", edgecolor="black")
ax.axhline(0, color="gray", lw=0.5, ls="--")
ax.axvline(0, color="gray", lw=0.5, ls="--")
ax.set_xlabel("East-West (m)")
ax.set_ylabel("North-South (m)")
ax.set_title(f"Nobeyama Radioheliograph T-array (N={len(antennas)} antennas)")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. u-v Coverage / u-v 덮개

**English.** For each antenna pair $(i, j)$ the baseline vector is $\mathbf{b}_{ij} = \mathbf{r}_i - \mathbf{r}_j$. In units of wavelengths the u-v coordinate is $(u, v) = \mathbf{b}_{ij} / \lambda$. We plot the instantaneous (snapshot) u-v coverage; Earth rotation would smear each point along an arc over time.

**한국어.** 각 안테나 쌍 $(i, j)$에 대해 기저선 벡터는 $\mathbf{b}_{ij} = \mathbf{r}_i - \mathbf{r}_j$이다. 파장 단위로 u-v 좌표는 $(u, v) = \mathbf{b}_{ij} / \lambda$이다. 순간 (스냅샷) u-v 덮개를 그리며; 지구 자전은 시간에 걸쳐 각 점을 호로 번지게 할 것이다.

In [ ]:
def compute_uv(antennas, wavelength):
    """Compute u-v samples for all antenna pairs at a given wavelength.

    Args:
        antennas: Nx2 antenna positions in meters.
        wavelength: Observing wavelength in meters.

    Returns:
        Mx2 array of (u, v) samples in units of wavelengths; M = N*(N-1).
        Both (b) and (-b) are included to reflect the Hermitian symmetry of the
        visibility function.
    """
    n = len(antennas)
    idx_i, idx_j = np.triu_indices(n, k=1)
    baselines = (antennas[idx_i] - antennas[idx_j]) / wavelength
    uv = np.vstack([baselines, -baselines])
    return uv


uv = compute_uv(antennas, LAMBDA_17)
print(f"Total u-v samples: {len(uv)} (incl. Hermitian conjugate)")
print(f"Max |u|: {np.max(np.abs(uv[:, 0])):.0f} wavelengths")
print(f"Max |v|: {np.max(np.abs(uv[:, 1])):.0f} wavelengths")

fig, ax = plt.subplots(1, 1, figsize=(8, 7))
ax.scatter(uv[:, 0], uv[:, 1], s=3, c="#2ca02c", alpha=0.5)
ax.set_xlabel("u (wavelengths)")
ax.set_ylabel("v (wavelengths)")
ax.set_title("NoRH snapshot u-v coverage at 17 GHz")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Synthesized Beam (Dirty Beam / PSF) / 합성 빔 (Dirty Beam / PSF)

**English.** The dirty beam is the inverse Fourier transform of the u-v sampling function. We grid the u-v samples onto a regular grid and IFFT to obtain the PSF. The FWHM of the central lobe gives the effective angular resolution.

**한국어.** Dirty 빔은 u-v 샘플링 함수의 역 푸리에 변환이다. u-v 샘플을 정규 격자에 gridding하고 IFFT하여 PSF를 얻는다. 중심 로브의 FWHM이 유효 각분해능을 준다.

In [ ]:
def grid_and_psf(uv, grid_size=512, pixel_arcsec=2.0):
    """Grid u-v samples and compute the dirty beam (PSF) via IFFT.

    Args:
        uv: Mx2 u-v samples in wavelengths.
        grid_size: Number of pixels per side of the image grid.
        pixel_arcsec: Angular pixel scale in arcseconds.

    Returns:
        Tuple of (psf, sampling_grid) where psf is the real-valued synthesized beam image
        and sampling_grid is the complex u-v sampling function used.
    """
    pixel_rad = pixel_arcsec / 206265.0
    delta_u = 1.0 / (grid_size * pixel_rad)
    grid = np.zeros((grid_size, grid_size), dtype=complex)

    u_idx = np.round(uv[:, 0] / delta_u).astype(int) + grid_size // 2
    v_idx = np.round(uv[:, 1] / delta_u).astype(int) + grid_size // 2

    mask = (u_idx >= 0) & (u_idx < grid_size) & (v_idx >= 0) & (v_idx < grid_size)
    for ui, vi in zip(u_idx[mask], v_idx[mask]):
        grid[vi, ui] += 1.0

    psf = np.real(fftshift(ifft2(fftshift(grid))))
    psf = psf / np.max(psf)
    return psf, grid


PIX_ARCSEC = 2.0
GRID = 512
psf, uv_grid = grid_and_psf(uv, grid_size=GRID, pixel_arcsec=PIX_ARCSEC)

extent = np.array([-GRID / 2, GRID / 2, -GRID / 2, GRID / 2]) * PIX_ARCSEC
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(psf, extent=extent, origin="lower", cmap="seismic", vmin=-0.3, vmax=1)
axes[0].set_title("Dirty beam (PSF) — 17 GHz snapshot")
axes[0].set_xlabel("arcsec")
axes[0].set_ylabel("arcsec")
axes[0].set_xlim(-100, 100)
axes[0].set_ylim(-100, 100)

center = GRID // 2
axs_x = (np.arange(GRID) - center) * PIX_ARCSEC
axes[1].plot(axs_x, psf[center, :], label="E-W cut", lw=1.5)
axes[1].plot(axs_x, psf[:, center], label="N-S cut", lw=1.5)
axes[1].set_xlim(-60, 60)
axes[1].set_xlabel("Offset (arcsec)")
axes[1].set_ylabel("Normalized PSF")
axes[1].set_title("PSF cross-sections")
axes[1].axhline(0.5, color="gray", lw=0.5, ls="--")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

cut = psf[center, :]
above_half = np.where(cut > 0.5)[0]
fwhm_ew = (above_half[-1] - above_half[0]) * PIX_ARCSEC
print(f"Approximate FWHM (E-W): {fwhm_ew:.1f} arcsec (paper target: 10 arcsec)")

## 5. Synthetic NoRH Flare Image / 합성 NoRH 플레어 영상

**English.** We construct a ground-truth sky model with two compact Gaussian flare sources embedded in a smooth solar disk, FFT to get the visibilities, sample onto the NoRH u-v coverage, and IFFT to get the dirty image. This mimics NoRH's snapshot imaging.

**한국어.** 부드러운 태양 원반에 두 개의 콤팩트 가우시안 플레어 광원을 삽입한 정답 천구 모델을 구성하고, FFT로 가시도를 얻고, NoRH u-v 덮개에 샘플링하고, IFFT로 dirty 영상을 얻는다. 이는 NoRH의 스냅샷 영상화를 모방한다.

In [ ]:
def make_sky_model(grid_size=512, pixel_arcsec=2.0):
    """Create a synthetic solar sky model: disk plus two Gaussian flare sources.

    Args:
        grid_size: Grid size in pixels.
        pixel_arcsec: Pixel scale in arcseconds.

    Returns:
        2D numpy array of brightness temperature in Kelvin.
    """
    y, x = np.indices((grid_size, grid_size)) - grid_size / 2
    y = y * pixel_arcsec
    x = x * pixel_arcsec
    r = np.sqrt(x**2 + y**2)

    R_SUN = 960.0
    EDGE = 20.0
    disk = 1.0e4 * 0.5 * (1.0 - np.tanh((r - R_SUN) / EDGE))

    flare1 = 1.0e7 * np.exp(-((x - 200) ** 2 + (y - 100) ** 2) / (2 * 15**2))
    flare2 = 5.0e6 * np.exp(-((x - 230) ** 2 + (y - 70) ** 2) / (2 * 10**2))

    return disk + flare1 + flare2


def simulate_observation(sky, uv, pixel_arcsec):
    """Forward-model NoRH observation: FFT -> sample -> IFFT.

    Args:
        sky: True sky brightness image (2D array).
        uv: u-v samples in wavelengths.
        pixel_arcsec: Pixel scale in arcsec.

    Returns:
        Dirty image (2D array, real-valued).
    """
    grid_size = sky.shape[0]
    pixel_rad = pixel_arcsec / 206265.0
    delta_u = 1.0 / (grid_size * pixel_rad)

    visibilities = fftshift(fft2(fftshift(sky)))
    sampling = np.zeros_like(visibilities)

    u_idx = np.round(uv[:, 0] / delta_u).astype(int) + grid_size // 2
    v_idx = np.round(uv[:, 1] / delta_u).astype(int) + grid_size // 2
    mask = (u_idx >= 0) & (u_idx < grid_size) & (v_idx >= 0) & (v_idx < grid_size)

    for ui, vi in zip(u_idx[mask], v_idx[mask]):
        sampling[vi, ui] = 1.0

    sampled_vis = visibilities * sampling
    dirty = np.real(fftshift(ifft2(fftshift(sampled_vis))))
    return dirty


sky = make_sky_model(grid_size=GRID, pixel_arcsec=PIX_ARCSEC)
dirty = simulate_observation(sky, uv, PIX_ARCSEC)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im0 = axes[0].imshow(np.log10(np.maximum(sky, 1)), extent=extent, origin="lower", cmap="afmhot")
axes[0].set_title("Ground-truth sky model (log $T_B$, K)")
axes[0].set_xlabel("arcsec")
axes[0].set_ylabel("arcsec")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(dirty, extent=extent, origin="lower", cmap="afmhot")
axes[1].set_title("NoRH dirty image (forward-modeled)")
axes[1].set_xlabel("arcsec")
axes[1].set_ylabel("arcsec")
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

## 6. Gyrosynchrotron Spectra at 17 and 34 GHz / 17 및 34 GHz 자이로싱크로트론 스펙트럼

**English.** Using the Dulk & Marsh (1982) simplified formulas, we compute the brightness temperature spectrum of a gyrosynchrotron source with varying magnetic field $B$ and electron power-law index $\delta$. The ratio $T_B(34)/T_B(17)$ directly reveals $\delta$.

**한국어.** Dulk & Marsh (1982) 단순화 공식을 사용하여, 다양한 자기장 $B$와 전자 멱법칙 지수 $\delta$를 가진 자이로싱크로트론 광원의 밝기 온도 스펙트럼을 계산한다. 비율 $T_B(34)/T_B(17)$는 $\delta$를 직접 드러낸다.

In [ ]:
def gyrosynchrotron_spectrum(nu_hz, B_gauss, delta, n_nonthermal=1e7, L_cm=1e9, theta_deg=45.0):
    """Approximate gyrosynchrotron brightness temperature using Dulk & Marsh (1982) scaling.

    Uses the empirical scaling where the spectrum peaks at a turnover frequency nu_peak
    separating an optically thick rising portion (~ nu^2.5) from an optically thin falling
    portion (~ nu^alpha_thin) with alpha_thin = -(1.22 - 0.90*delta).

    Args:
        nu_hz: Frequencies in Hz (array).
        B_gauss: Magnetic field strength in Gauss.
        delta: Electron power-law index (typical 2-5).
        n_nonthermal: Nonthermal electron density in cm^-3.
        L_cm: Source path length in cm.
        theta_deg: Angle between magnetic field and line of sight in degrees.

    Returns:
        Tuple of (T_B array in K, nu_peak in Hz).
    """
    theta_rad = np.deg2rad(theta_deg)
    sin_t = max(np.sin(theta_rad), 0.1)

    nu_peak_mhz = (
        2.72e3
        * 10 ** (0.27 * delta)
        * sin_t ** (0.41 + 0.03 * delta)
        * (n_nonthermal * L_cm) ** (0.32 - 0.03 * delta)
        * B_gauss ** (0.68 + 0.03 * delta)
    )
    nu_peak_hz = nu_peak_mhz * 1e6

    alpha_thin = -(1.22 - 0.90 * delta)
    T_peak = 1e8

    T_B = np.where(
        nu_hz < nu_peak_hz,
        T_peak * (nu_hz / nu_peak_hz) ** 2.5,
        T_peak * (nu_hz / nu_peak_hz) ** alpha_thin,
    )
    return T_B, nu_peak_hz


nu_arr = np.logspace(9, 11.3, 400)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for B in [100, 300, 500, 1000]:
    T_B, nu_peak = gyrosynchrotron_spectrum(nu_arr, B_gauss=B, delta=3.0)
    axes[0].loglog(nu_arr / 1e9, T_B, label=f"B = {B} G, nu_pk = {nu_peak/1e9:.1f} GHz")
axes[0].axvline(17, color="red", ls="--", alpha=0.7, label="17 GHz")
axes[0].axvline(34, color="blue", ls="--", alpha=0.7, label="34 GHz")
axes[0].set_xlabel("Frequency (GHz)")
axes[0].set_ylabel("Brightness temperature (K)")
axes[0].set_title("Gyrosynchrotron spectrum vs. B (delta=3)")
axes[0].legend(fontsize=8)
axes[0].grid(True, which="both", alpha=0.3)

for d in [2.0, 3.0, 4.0, 5.0]:
    T_B, nu_peak = gyrosynchrotron_spectrum(nu_arr, B_gauss=300, delta=d)
    axes[1].loglog(nu_arr / 1e9, T_B, label=f"delta = {d}, nu_pk = {nu_peak/1e9:.1f} GHz")
axes[1].axvline(17, color="red", ls="--", alpha=0.7, label="17 GHz")
axes[1].axvline(34, color="blue", ls="--", alpha=0.7, label="34 GHz")
axes[1].set_xlabel("Frequency (GHz)")
axes[1].set_ylabel("Brightness temperature (K)")
axes[1].set_title("Gyrosynchrotron spectrum vs. delta (B=300 G)")
axes[1].legend(fontsize=8)
axes[1].grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Two-frequency Ratio for Electron Energy Retrieval / 전자 에너지 추정을 위한 2주파 비율

**English.** The ratio $T_B(34\,\text{GHz}) / T_B(17\,\text{GHz})$ in the optically thin regime depends strongly on the electron power-law index $\delta$, providing a clean diagnostic. Here we compute the ratio for a grid of $(B, \delta)$.

**한국어.** 광학적으로 얇은 영역에서 비율 $T_B(34\,\text{GHz}) / T_B(17\,\text{GHz})$는 전자 멱법칙 지수 $\delta$에 강하게 의존하여, 깔끔한 진단을 제공한다. 여기서 $(B, \delta)$ 격자에 대해 비율을 계산한다.

In [ ]:
B_range = np.linspace(100, 1200, 40)
delta_range = np.linspace(2.0, 5.0, 30)
BB, DD = np.meshgrid(B_range, delta_range, indexing="xy")
ratio = np.zeros_like(BB)

for i in range(BB.shape[0]):
    for j in range(BB.shape[1]):
        T17, _ = gyrosynchrotron_spectrum(np.array([17e9]), B_gauss=BB[i, j], delta=DD[i, j])
        T34, _ = gyrosynchrotron_spectrum(np.array([34e9]), B_gauss=BB[i, j], delta=DD[i, j])
        ratio[i, j] = T34[0] / T17[0]

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.pcolormesh(BB, DD, np.log10(ratio), cmap="viridis", shading="auto")
cs = ax.contour(BB, DD, ratio, levels=[0.1, 0.2, 0.3, 0.5, 1.0, 2.0], colors="white")
ax.clabel(cs, inline=True, fontsize=9)
ax.set_xlabel("Magnetic field B (G)")
ax.set_ylabel("Electron power-law index delta")
ax.set_title("log10[ T_B(34) / T_B(17) ] across (B, delta) grid")
plt.colorbar(im, ax=ax, label="log10 ratio")
plt.tight_layout()
plt.show()

## 8. 1D Profile of the Solar Disk at 17 GHz / 17 GHz 태양 원반의 1D 프로파일

**English.** A cross-section through the center of the solar disk at 17 GHz shows the roughly flat $T_B \sim 10^4$ K quiet-Sun chromospheric level, enhanced $\sim 10^5$ K active regions, and the sharp limb (2.5% larger than the optical disk due to chromospheric opacity). This illustrates why the modified CLEAN first subtracts the disk before extracting flares.

**한국어.** 17 GHz에서 태양 원반 중심을 통과하는 단면은 대략 평평한 $T_B \sim 10^4$ K 정온태양 채층 수준, 증강된 $\sim 10^5$ K 활동영역, 그리고 날카로운 주변부 (채층 불투명도로 인해 광학 원반보다 2.5% 크다)를 보여준다. 이는 수정 CLEAN이 플레어를 추출하기 전에 먼저 원반을 차감하는 이유를 보여준다.

In [ ]:
def solar_disk_1d(x_arcsec, R_sun_arcsec=984.0, T_quiet=1e4, T_limb_bright=2e4, edge=15.0):
    """Simple 1D model of brightness temperature across the solar disk at 17 GHz.

    Includes a flat disk interior, limb brightening, and a smooth chromospheric edge.
    The radio disk radius is 2.5% larger than the optical disk (R_optical = 960 arcsec)
    because the tau=1 surface for 17 GHz free-free emission lies in the chromosphere.

    Args:
        x_arcsec: 1D coordinate array in arcsec.
        R_sun_arcsec: Radio disk radius in arcsec (default 984 = 1.025 * 960).
        T_quiet: Quiet-Sun disk center brightness temperature (K).
        T_limb_bright: Peak limb-brightening brightness temperature (K).
        edge: Width of the chromospheric edge in arcsec.

    Returns:
        Brightness temperature array in K.
    """
    r = np.abs(x_arcsec)
    disk = T_quiet * 0.5 * (1.0 - np.tanh((r - R_sun_arcsec) / edge))
    limb = T_limb_bright * np.exp(-((r - (R_sun_arcsec - 20)) ** 2) / (2 * 15.0**2))
    limb = limb * (r < R_sun_arcsec)
    AR = 1.5e5 * np.exp(-((x_arcsec - 250) ** 2) / (2 * 40.0**2))
    return disk + limb + AR


x = np.linspace(-1100, 1100, 2000)
T_profile = solar_disk_1d(x)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(x, T_profile, lw=1.5)
ax.axvline(-960, color="gray", ls=":", label="optical limb")
ax.axvline(960, color="gray", ls=":")
ax.axvline(-984, color="red", ls="--", alpha=0.5, label="17 GHz radio limb (+2.5%)")
ax.axvline(984, color="red", ls="--", alpha=0.5)
ax.set_yscale("log")
ax.set_xlabel("Offset from disk center (arcsec)")
ax.set_ylabel("Brightness temperature T_B (K)")
ax.set_title("Solar disk brightness at 17 GHz (1D cross-section)")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Summary / 요약

**English.** This notebook reproduced the key interferometric physics of the Nobeyama Radioheliograph:

1. **Array geometry**: A T-shaped array of 84 antennas with geometric-progression spacings produces dense central u-v sampling plus long baselines for 10 arcsec resolution.
2. **u-v coverage and PSF**: The snapshot u-v coverage yields a synthesized beam (dirty beam / PSF) whose main lobe sets the resolution and whose sidelobes must be CLEANed.
3. **Forward imaging**: A synthetic sky with a disk plus flare sources, FFT-transformed and sampled onto the NoRH u-v coverage, produces a realistic dirty image showing both the bright flares and the heavy sidelobe contamination from the disk.
4. **Gyrosynchrotron diagnostics**: At 17 and 34 GHz the two-frequency spectral ratio directly constrains the electron power-law index $\delta$; at 17 GHz alone the brightness temperature is most sensitive to $B$ and optical depth.
5. **Solar disk profile**: The 17 GHz radio disk is 2.5% larger than the optical disk, with active-region enhancements up to $\sim 10^5$ K, explaining why NoRH's modified CLEAN first subtracts the disk before deconvolving flare sources.

**한국어.** 이 노트북은 Nobeyama Radioheliograph의 핵심 간섭계 물리를 재현했다:

1. **배열 기하**: 기하급수적 간격을 가진 84개 안테나의 T자형 배열은 조밀한 중심 u-v 샘플링과 10 arcsec 분해능을 위한 긴 기저선을 생성한다.
2. **u-v 덮개와 PSF**: 스냅샷 u-v 덮개는 주엽이 분해능을 설정하고 사이드로브가 CLEAN되어야 하는 합성 빔 (dirty beam / PSF)을 만들어낸다.
3. **순방향 영상화**: 원반과 플레어 광원을 가진 합성 천구를 FFT 변환하고 NoRH u-v 덮개에 샘플링하여, 밝은 플레어와 원반의 무거운 사이드로브 오염을 모두 보여주는 현실적 dirty 영상을 생성한다.
4. **자이로싱크로트론 진단**: 17과 34 GHz에서 2주파 스펙트럼 비율이 전자 멱법칙 지수 $\delta$를 직접 제약하며; 17 GHz 단독으로는 밝기 온도가 $B$와 광학 깊이에 가장 민감하다.
5. **태양 원반 프로파일**: 17 GHz 전파 원반은 광학 원반보다 2.5% 크고, 활동영역 증강이 $\sim 10^5$ K에 달하여, NoRH의 수정 CLEAN이 플레어 광원을 디컨볼루션하기 전에 먼저 원반을 차감하는 이유를 설명한다.

---

### References / 참고문헌

- Nakajima, H. et al., Proc. IEEE 82(5), 705-713 (1994).
- Dulk, G. A., Marsh, K. A., Astrophys. J. 259, 350 (1982).
- Dulk, G. A., Annu. Rev. Astron. Astrophys. 23, 169 (1985).
- Bastian, T. S., Benz, A. O., Gary, D. E., ARA&A 36, 131 (1998). [Paper #26]